# Apigee Template: REST-AI-Interactions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-Interactions.ipynb)

**Template Name:** `REST-AI-Interactions`  
**Description:** Google Gemini Interactions API Gateway proxy supporting multi-turn conversational agents, session state management, model routing, model checks, and token usage analytics.

### Key Capabilities:
- **Model Pre-Processing & Checks:** Apigee inspects the incoming request (`JS-CheckModel`), validates requested models, and performs model mapping/aliasing (e.g., mapping `gemini-flash-latest` to `gemini-3.7-flash`).
- **Token Usage Analytics & Data Collection:** On response delivery, Apigee extracts token consumption metrics and records them directly into Apigee Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`, `dc_ai_model`). These metrics feed Apigee Custom Reports for real-time cost, latency, and consumption tracking.
- **Server-Managed Conversation State:** Seamlessly forwards `previous_interaction_id` to the Gemini Interactions API to maintain multi-turn conversational context without requiring clients to send full chat histories.

### Documentation & References:
- [Google Gemini Interactions API Documentation](https://ai.google.dev/api/interactions)
- [Apigee Feature Templater (aft) GitHub](https://github.com/apigee/apigee-templater)
- [Apigee Data Collectors Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/data-collectors)
- [Apigee Custom Reports Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/reports-overview)

### Workflow:
1. **Configuration & Authentication:** Enter your `GOOGLE_CLOUD_PROJECT`, `APIGEE_ENV`, and `GEMINI_API_KEY`.
2. **Setup, Initialize Resources & Deploy:** Install `aft`, run `sh/initialize.sh` (service account, IAM roles, data collectors, reports), and deploy `REST-AI-Interactions.yaml`.
3. **Test Initialized Interactions API:** Send single-turn, alias mapping, and multi-turn requests using `APIGEE_HOST`.

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID (Apigee Organization), Apigee environment, and your Gemini API Key.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
GOOGLE_CLOUD_PROJECT = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
GEMINI_API_KEY = "your_gemini_api_key"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ORG"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ENV"] = APIGEE_ENV
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["APIGEE_SA"] = f"apigee-service@{GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {GOOGLE_CLOUD_PROJECT}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Downloads `templates/REST-AI-Interactions.yaml` and `sh/initialize.sh` (if running standalone in Colab), installs `aft`, runs resource initialization (service accounts, IAM bindings, data collectors, custom reports), sets `APIGEE_HOST`, and deploys the template.

In [ ]:
# @title 2. Setup, Initialize & Deploy Template
import os
import subprocess

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-Interactions.yaml" if os.path.exists("REST-AI-Interactions.yaml") else "templates/REST-AI-Interactions.yaml" if os.path.exists("templates/REST-AI-Interactions.yaml") else "REST-AI-Interactions.yaml"
os.environ["TEMPLATE_FILE"] = TEMPLATE_FILE

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-Interactions.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization script (sets up service account, IAM bindings, data collectors, reports)
!bash sh/initialize.sh

# 4. Resolve APIGEE_HOST using aft describe
cmd = 'aft describe --project "$GOOGLE_CLOUD_PROJECT" -f json | jq --raw-output ".environmentGroups[] | select(any(.attachments[]; .environment == \"$APIGEE_ENV\")) | .hostnames[0]"'
try:
    host = subprocess.check_output(cmd, shell=True, text=True).strip()
    if host and host != "null":
        os.environ["APIGEE_HOST"] = host
        print(f"APIGEE_HOST resolved to: {host}")
except Exception as e:
    print(f"Could not automatically resolve APIGEE_HOST via aft: {e}")

# 5. Deploy template with aft
!aft "$TEMPLATE_FILE" \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --env="$APIGEE_ENV" \
  --sa="$APIGEE_SA"


## 3. Test Initialized API via APIGEE_HOST

Send requests to `https://${APIGEE_HOST}/v1beta/interactions`.

- **Model Checks:** The proxy validates the requested model and applies model alias mapping.
- **Token Usage Recording:** Apigee captures prompt and candidate token counts into Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`).
- **Interactions API Protocol:** Supports single-turn interactions and multi-turn stateful chaining via `previous_interaction_id`.

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import json
import requests

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
APIGEE_ENV = os.getenv("APIGEE_ENV", "dev")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Determine APIGEE_HOST
APIGEE_HOST = os.getenv("APIGEE_HOST")
if not APIGEE_HOST or APIGEE_HOST == "null":
    APIGEE_HOST = f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net"
    os.environ["APIGEE_HOST"] = APIGEE_HOST

ENDPOINT_URL = f"https://{APIGEE_HOST}/v1beta/interactions"
print(f"APIGEE_HOST: {APIGEE_HOST}")
print(f"Interactions Endpoint: {ENDPOINT_URL}")

def send_interaction(model: str, input_text: str, previous_interaction_id: str = None):
    """
    Sends a request to the Gemini Interactions API through the Apigee Gateway.
    Apigee applies model checks/mapping and logs token usage into Apigee Data Collectors.
    """
    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": GEMINI_API_KEY
    }
    
    payload = {
        "model": model,
        "input": input_text
    }
    if previous_interaction_id:
        payload["previous_interaction_id"] = previous_interaction_id

    print(f"\n---> Sending [{model}] interaction to {ENDPOINT_URL}...")
    try:
        url = f"{ENDPOINT_URL}?key={GEMINI_API_KEY}" if GEMINI_API_KEY else ENDPOINT_URL
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        print(f"HTTP Status: {response.status_code}")
        
        try:
            data = response.json()
            print(json.dumps(data, indent=2))
            
            # Display token usage recorded by Apigee
            usage = data.get("usage", data.get("usageMetadata", {}))
            if usage:
                print("\n[Token Usage Recorded by Apigee Data Collectors]")
                print(f"  Prompt Tokens:   {usage.get('promptTokenCount', usage.get('input_tokens', 'N/A'))}")
                print(f"  Response Tokens: {usage.get('candidatesTokenCount', usage.get('output_tokens', 'N/A'))}")
                print(f"  Total Tokens:    {usage.get('totalTokenCount', usage.get('total_tokens', 'N/A'))}")
            return data
        except Exception:
            print(response.text)
            return None
    except Exception as e:
        print("Request error:", e)
        return None


In [ ]:
# @title Test 1: Single-Turn Interaction (gemini-2.5-flash)
# Apigee verifies model parameters, routes to Gemini, and captures token analytics.
resp1 = send_interaction(
    model="gemini-2.5-flash",
    input_text="Explain how an API gateway provides governance for enterprise AI in two sentences."
)


In [ ]:
# @title Test 2: Model Checks & Alias Mapping (gemini-flash-latest)
# The proxy checks the model and applies ModelMapping (gemini-flash-latest -> gemini-3.7-flash).
resp2 = send_interaction(
    model="gemini-flash-latest",
    input_text="What are 3 critical metrics to track when operating production LLM applications?"
)


In [ ]:
# @title Test 3: Multi-Turn Conversation with previous_interaction_id
# Uses server-managed session state from Test 1 without re-sending earlier turns.
prev_id = None
if resp1 and "id" in resp1:
    prev_id = resp1["id"]
    print(f"Chaining from previous interaction ID: {prev_id}")

resp3 = send_interaction(
    model="gemini-2.5-flash",
    input_text="Summarize your earlier explanation in exactly 5 words.",
    previous_interaction_id=prev_id
)


## 4. Verify Apigee Analytics & Reports

Because `sh/initialize.sh` provisioned Data Collectors and Custom Reports, every request processed by `REST-AI-Interactions` records token analytics:

1. Open the [Google Cloud Apigee Console](https://console.cloud.google.com/apigee).
2. Navigate to **Analytics > Custom Reports**.
3. Inspect the pre-configured reports:
   - **`ai_token_cost_by_model_user`**: Aggregated AI token cost by model and user/developer.
   - **`ai_model_usage_latency`**: Real-time latency and call volume per model.
   - **`ai_token_counts_by_model_user`**: Prompt, response, and total token count breakdowns.
